In [1]:
# imports
import numpy as np
import matplotlib.pyplot as plt
import time
import matplotlib.ticker as ticker
import pandas as pd
import scipy.sparse as sp
import scipy.sparse.linalg as spla


from chebyshev_hofbauer_resonances.general_tent_map.approx_transfer_op import (
    approx_super_adjacency,
    approx_ulams,
)

In [2]:
# function definition
epsilon = 0.1
function_domains = [(0, 0.5), (0.5, 1)]
functions = [lambda x: epsilon + x * (1 + (2 - 4 * epsilon) * x), lambda x: 2 * x - 1]
inverses = [
    lambda y: (
        (-1 + np.sqrt((1 - 4 * epsilon) ** 2 + (8 - 16 * epsilon) * y))
        / (4 - 8 * epsilon)
    ),
    lambda y: (1 + y) / 2,
]
derivatives = [lambda x: 1 + (4 - 8 * epsilon) * x, lambda _: 2]

In [3]:
# functions
def super_adj_spec(NK, depth, spec_size = 3):
    N = K = NK

    super_adjacency = approx_super_adjacency(
        function_domains,
        functions,
        inverses,
        derivatives,
        N=N,
        K=K,
        depth=depth,
    )


    sparse_adj = sp.csr_matrix(super_adjacency)
    evals_super_adj = spla.eigs(sparse_adj, k=2, which='LM', return_eigenvectors=False)
    evals_super_adj = evals_super_adj[np.argsort(-np.abs(evals_super_adj))]

    eval_mags = np.abs(evals_super_adj)

    spec = eval_mags[:spec_size]
    return spec, eval_mags



def ulam_spec(N, M, spec_size=3):
    L_ulam = approx_ulams(
        function_domains,
        functions,
        inverses,
        derivatives,
        N=N,
        M=M,
    )

    evals_ulam = np.linalg.eigvals(L_ulam)
    evals_ulam = evals_ulam[np.argsort(-np.abs(evals_ulam))]

    eval_mags = np.abs(evals_ulam)
    eval_mags = np.sort(eval_mags)[::-1]

    spec = eval_mags[:spec_size]
    return spec, evals_ulam

In [8]:
spec, evals_super_adj = super_adj_spec(NK=50, depth=200, spec_size = 2)

KeyboardInterrupt: 